# 03 — Modèle ML : Gradient Boosted Trees + SHAP sur les faits de jeu

Objectif : entraîner un modèle qui prédit la victoire à partir des faits de jeu (objectifs, combat, early game...) réalisés par chaque champion, puis l'expliquer avec SHAP pour répondre à trois questions : quels faits comptent pour gagner en général, lesquels comptent pour un champion donné, et lesquels priment pour une composition donnée.

**Mise à jour** : `data/matches.db` contient maintenant **12 482 matchs** (collecte à grande échelle terminée le 2026-09-15/16) — le pipeline ci-dessous a été validé d'abord sur un petit échantillon (~20 matchs), puis rejoué tel quel sur les vraies données.

**Partie 1 : extraire un dataset tabulaire propre depuis SQLite.**

## Concept 1 — Charger les matchs bruts

On réutilise `lol_assistant.database.get_connection` (même base que la phase 2), puis `pd.read_sql_query` pour récupérer toutes les lignes de la table `matches` sous forme de `DataFrame` — chaque ligne a `match_id`, `region`, `raw_json` (le JSON complet du match, tel que stocké par le crawler).

In [1]:
import pandas as pd

from lol_assistant.database import get_connection

conn = get_connection("../data/matches.db")

raw_df = pd.read_sql_query("SELECT match_id, raw_json FROM matches", conn)
print("Nombre de matchs bruts:", len(raw_df))
raw_df.head()

Nombre de matchs bruts: 12482


,match_id,raw_json
0,EUW1_7981560267,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
1,EUW1_7981526567,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
2,EUW1_7979271688,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
3,EUW1_7978928820,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
4,EUW1_7978885105,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."


## Concept 2 — Extraire une ligne structurée par match

`raw_json` est un gros bloc de texte JSON — on le parse avec `json.loads`, puis on isole ce dont on a besoin : la liste des 5 champions de l'équipe bleue, celle des 5 champions de l'équipe rouge, si l'équipe bleue a gagné (`teamId == 100`), et `gameDuration` (utile juste après pour repérer les remakes). On garde aussi `gameMode` — utile pour un autre nettoyage qu'on va faire juste après (exclure les modes de jeu qui ne sont pas le 5v5 classique).

In [2]:
import json


def extract_match_row(raw_json):
    match_data = json.loads(raw_json)
    info = match_data["info"]

    blue_champions = []
    red_champions = []
    blue_win = None

    for p in info["participants"]:
        if p["teamId"] == 100:
            blue_champions.append(p["championName"])
            blue_win = p["win"]
        else:
            red_champions.append(p["championName"])

    return {
        "blue_champions": blue_champions,
        "red_champions": red_champions,
        "blue_win": blue_win,
        "game_duration": info["gameDuration"],
        "game_mode": info["gameMode"],
    }


rows = [extract_match_row(raw) for raw in raw_df["raw_json"]]
matches_df = pd.DataFrame(rows)
matches_df["match_id"] = raw_df["match_id"]

matches_df.head()

,blue_champions,red_champions,blue_win,game_duration,game_mode,match_id
0,"[Jax, Shyvana, Leblanc, Yunara, Leona]","[Camille, Sylas, Zed, Caitlyn, Morgana]",True,1141,CLASSIC,EUW1_7981560267
1,"[Zilean, Skarner, Tristana, Viktor, Sona]","[Pantheon, Belveth, Riven, Syndra, Elise]",True,1773,CLASSIC,EUW1_7981526567
2,"[Rumble, Sylas, Ahri, Ezreal, Nautilus]","[Aurora, Talon, Zed, Pantheon, Seraphine]",False,2488,CLASSIC,EUW1_7979271688
3,"[Pantheon, Qiyana, AurelionSol, Jhin, Elise]","[Tryndamere, Zyra, Ahri, Sivir, Zilean]",False,1494,CLASSIC,EUW1_7978928820
4,"[Jayce, Skarner, Ekko, Zoe, Thresh]","[Pantheon, LeeSin, Lux, Tristana, Nautilus]",True,2377,CLASSIC,EUW1_7978885105


## Concept 3 — Nettoyage : repérer les remakes

Un "remake" (partie annulée tôt, ex: un joueur absent dès le début, remboursée par Riot) n'est pas un vrai résultat de composition — l'inclure fausserait l'entraînement (une victoire/défaite qui ne reflète rien de stratégique). On repère ça via `gameDuration`, anormalement bas pour une vraie partie.

On inspecte d'abord la distribution avant de choisir un seuil, plutôt que d'en deviner un au hasard — et on vérifie au passage l'unité (secondes ou millisecondes, Riot a changé ça selon les versions de patch).

In [3]:
matches_df["game_duration"].describe()

count    12482.000000
mean      1633.152780
std        409.425716
min         64.000000
25%       1416.000000
50%       1643.000000
75%       1886.000000
max       3488.000000
Name: game_duration, dtype: float64

**À vérifier en exécutant** : si les valeurs sont de l'ordre de quelques centaines à ~3000, `gameDuration` est en **secondes** (une partie dure typiquement 15-40 min, soit 900-2400s) — un seuil de remake autour de 300s (5 min) est raisonnable. Si les valeurs sont de l'ordre du million, c'est en **millisecondes**, et il faudra diviser par 1000 avant de filtrer. On ajustera le code juste après selon ce qu'on observe.

In [4]:
REMAKE_THRESHOLD_SECONDS = 300  # confirmé : gameDuration est en secondes sur cet échantillon

n_before = len(matches_df)
matches_df = matches_df[matches_df["game_duration"] >= REMAKE_THRESHOLD_SECONDS].reset_index(drop=True)
n_after = len(matches_df)

print(f"Matchs retirés (probables remakes) : {n_before - n_after}")
print(f"Matchs restants : {n_after}")

Matchs retirés (probables remakes) : 132
Matchs restants : 12350


## Concept 3bis — Nettoyage : exclure les modes de jeu hors-sujet

Le crawler snowball explore les matchs récents des joueurs croisés — sans distinction de mode. Résultat concret sur nos données : certains matchs sont en **Arena** (`gameMode = "CHERRY"`, équipes de 2, pas de lanes, objectifs complètement différents), pas du 5v5 Summoner's Rift classique. Les inclure fausserait tout : ce ne sont pas des variations stratégiques d'un même jeu, c'est un jeu différent.

On ne garde que `gameMode == "CLASSIC"` (couvre normales, ranked solo/duo, ranked flex, Clash — tout le 5v5 classique). Le crawler (`lol_assistant/crawler.py`) a aussi été corrigé pour ne plus stocker ces matchs à l'avenir.

In [5]:
print("Modes de jeu présents:", matches_df["game_mode"].value_counts().to_dict())

n_before = len(matches_df)
matches_df = matches_df[matches_df["game_mode"] == "CLASSIC"].reset_index(drop=True)
n_after = len(matches_df)

print(f"Matchs retirés (hors 5v5 classique) : {n_before - n_after}")
print(f"Matchs restants : {n_after}")

Modes de jeu présents: {'CLASSIC': 12346, 'CHERRY': 4}
Matchs retirés (hors 5v5 classique) : 4
Matchs restants : 12346


---

**Checkpoint** : à ce stade, `matches_df` contient un dataset tabulaire propre (`blue_champions`, `red_champions`, `blue_win`, `match_id`), débarrassé des remakes éventuels.

**Mise à jour d'architecture (voir mémoire projet)** : on n'utilise plus une Factorization Machine sur les compositions seules — un problème mathématique réel (produit scalaire symétrique, incapable de coder une relation directionnelle "A contre B") nous a fait pivoter vers **Gradient Boosted Trees + SHAP** sur des **faits de jeu** (objectifs, kills, early game...), extraits du champ `challenges` déjà présent dans `Match-V5` (pas besoin de la Timeline).

Prochaine étape : extraire les faits de jeu depuis `challenges`, au niveau équipe.

## Concept 4 — Le champ `challenges` : une mine de faits de jeu déjà agrégés

Chaque participant d'un match `Match-V5` a un champ `challenges` avec ~125 statistiques dérivées, déjà calculées par Riot : objectifs (`dragonTakedowns`, `teamBaronKills`...), combat (`killParticipation`, `teamDamagePercentage`...), early game (`laningPhaseGoldExpAdvantage`...), vision... exactement le genre de "faits de jeu" qu'on voulait extraire des événements bruts, mais sans avoir à toucher à la Timeline.

On exclut d'abord ce qui vient à coup sûr d'autres modes de jeu (Arena `SWARM_*`, quelques mécaniques ARAM spécifiques) — le reste est gardé large, comme convenu.

Ensuite, un détail structurel important : certains champs sont **au niveau équipe** (répétés à l'identique chez les 5 joueurs, ex: `teamBaronKills`), d'autres **au niveau joueur** (varient, ex: `soloKills`). On garde les deux catégories **séparément identifiées**, mais **sans jamais sommer les champs individuels** — l'objectif étant justement de savoir quel fait de jeu compte pour quel champion précisément, sommer effacerait cette information. Les champs équipe restent un contexte partagé ; les champs joueur restent attachés au joueur (donc au champion) qui les a réalisés.

Plutôt que deviner à la main quelle catégorie s'applique à quel champ, on le **détecte depuis nos propres données** : si les 5 valeurs d'un champ sont toujours identiques au sein d'une équipe, sur tout le dataset, c'est un champ équipe ; sinon, un champ joueur.

**Avertissement** : sur seulement 20 matchs, cette classification est bruitée (un champ réellement "par joueur" peut sembler constant par coïncidence sur un petit échantillon). Elle se refera automatiquement, et deviendra fiable, une fois la vraie collecte lancée.

In [6]:
EXCLUDED_CHALLENGE_PREFIXES = ("SWARM_",)  # Arena
EXCLUDED_CHALLENGE_FIELDS = {"poroExplosions", "fistBumpParticipation", "snowballsHit"}  # ARAM


def is_relevant_challenge_field(field):
    if any(field.startswith(prefix) for prefix in EXCLUDED_CHALLENGE_PREFIXES):
        return False
    return field not in EXCLUDED_CHALLENGE_FIELDS


def to_hashable(value):
    return tuple(value) if isinstance(value, list) else value


team_constant_fields = set()
player_level_fields = set()
list_valued_fields = set()

for raw in raw_df["raw_json"]:
    match_data = json.loads(raw)
    participants = match_data["info"]["participants"]

    for team_id in (100, 200):
        team_players = [p for p in participants if p["teamId"] == team_id]
        challenge_keys = set()
        for p in team_players:
            challenge_keys |= set(p.get("challenges", {}).keys())

        for field in challenge_keys:
            if not is_relevant_challenge_field(field):
                continue

            raw_values = [p.get("challenges", {}).get(field) for p in team_players]
            if any(isinstance(v, list) for v in raw_values):
                list_valued_fields.add(field)
                continue

            values = {to_hashable(v) for v in raw_values}
            if len(values) == 1:
                team_constant_fields.add(field)
            else:
                player_level_fields.add(field)

# un champ n'est "équipe" que s'il n'a JAMAIS varié sur tout le dataset
team_constant_fields -= player_level_fields
team_constant_fields.discard("gameLength")  # doublon avec game_duration, déjà extrait

print("Champs équipe:", len(team_constant_fields))
print("Champs joueur (à sommer):", len(player_level_fields))
print("Champs à valeur liste:", sorted(list_valued_fields))

Champs équipe: 19
Champs joueur (à sommer): 108
Champs à valeur liste: ['legendaryItemUsed']


## Concept 5 — Une ligne par (match, joueur), pas par match

Contrairement à une première version où on agrégeait tout au niveau équipe (erreur corrigée ici), le dataset final doit avoir **une ligne par joueur dans un match**. Chaque ligne garde :
- `champion` : le champion joué sur cette ligne — devient une vraie feature du modèle, plus une simple clé d'agrégation.
- `role` (`teamPosition` côté Riot : `TOP`/`JUNGLE`/`MIDDLE`/`BOTTOM`/`UTILITY`) — **ajouté ici**, oublié dans une première version. Important : un support et un ADC ont des profils de stats radicalement différents (vision vs dégâts, par exemple) ; sans le rôle, le modèle et l'exploration Power BI ne peuvent pas distinguer "ce fait compte peu pour ce champion" de "ce fait compte peu pour ce rôle, indépendamment du champion".
- Les **champs joueur**, tels quels, non sommés — attachés à ce champion précis.
- Les **champs équipe**, comme contexte partagé (répétés pour les 5 lignes de la même équipe).
- `team_win` : est-ce que l'équipe de ce joueur a gagné.
- `game_duration` : repris de `matches_df` (déjà filtré des remakes), comme contexte supplémentaire.

Le champ à valeur liste (`legendaryItemUsed`) redevient simple : plus besoin de sommer sur l'équipe, juste le nombre d'objets légendaires utilisés par **ce** joueur (`len(liste)`).

In [7]:
def extract_player_row(participant, match_id, game_duration):
    challenges = participant.get("challenges", {})

    row = {
        "match_id": match_id,
        "team": "blue" if participant["teamId"] == 100 else "red",
        "champion": participant["championName"],
        "role": participant.get("teamPosition") or "UNKNOWN",
        "team_win": participant["win"],
        "game_duration": game_duration,
    }

    for field in team_constant_fields:
        row[field] = challenges.get(field)

    for field in player_level_fields:
        row[field] = challenges.get(field)

    for field in list_valued_fields:
        row[field] = len(challenges.get(field) or [])

    return row


# game_duration déjà filtré des remakes dans matches_df (Concept 3)
game_duration_by_match = dict(zip(matches_df["match_id"], matches_df["game_duration"]))
raw_json_by_match = dict(zip(raw_df["match_id"], raw_df["raw_json"]))

player_rows = []
for match_id, game_duration in game_duration_by_match.items():
    match_data = json.loads(raw_json_by_match[match_id])
    for participant in match_data["info"]["participants"]:
        player_rows.append(extract_player_row(participant, match_id, game_duration))

dataset_df = pd.DataFrame(player_rows)
print("Dataset final:", dataset_df.shape)
print("Rôles présents:", dataset_df["role"].value_counts().to_dict())
dataset_df.head()

Dataset final: (123460, 134)
Rôles présents: {'JUNGLE': 24692, 'MIDDLE': 24692, 'BOTTOM': 24691, 'UTILITY': 24691, 'TOP': 24690, 'UNKNOWN': 4}


,match_id,team,champion,role,team_win,game_duration,earliestElderDragon,perfectDragonSoulsTaken,twentyMinionsIn3SecondsCount,earliestBaron,...,multikillsAfterAggressiveFlash,dodgeSkillShotsSmallWindow,survivedSingleDigitHpCount,multikills,killParticipation,maxLevelLeadLaneOpponent,killAfterHiddenWithAlly,outerTurretExecutesBefore10Minutes,enemyJungleMonsterKills,legendaryItemUsed
0,EUW1_7981560267,blue,Jax,TOP,True,1141,NaN,0,0,NaN,...,0,0,0,0,0.153846,1.0,1,0,0,1
1,EUW1_7981560267,blue,Shyvana,JUNGLE,True,1141,NaN,0,0,NaN,...,0,0,0,0,0.307692,2.0,0,0,16,2
2,EUW1_7981560267,blue,Leblanc,MIDDLE,True,1141,NaN,0,0,NaN,...,1,0,0,1,0.730769,1.0,0,0,0,2
3,EUW1_7981560267,blue,Yunara,BOTTOM,True,1141,NaN,0,0,NaN,...,0,0,0,0,0.615385,1.0,1,0,0,2
4,EUW1_7981560267,blue,Leona,UTILITY,True,1141,NaN,0,0,NaN,...,0,0,0,0,0.692308,3.0,0,0,0,2


---

**Checkpoint** : `dataset_df` a maintenant **123 460 lignes** (12 346 matchs `CLASSIC` valides × 10 joueurs) et **133 colonnes**, sur la collecte à grande échelle du 2026-09-15/16. 173 champions distincts représentés, le moins fréquent avec 93 occurrences — largement suffisant pour la suite.

Avant l'entraînement du modèle : une étape d'exploration visuelle dans Power BI (voir section suivante), puis entraînement GBT + SHAP dans une prochaine session.

## Concept 6 — Export pour exploration visuelle dans Power BI

Avant l'entraînement du modèle, une étape d'exploration visuelle interactive dans **Power BI** (Desktop, sur une autre machine Windows) : regarder les taux de victoire par champion, les corrélations entre faits de jeu et victoire, la distribution des features, etc. — avant de se lancer dans le ML.

Power BI ne tourne pas sur cette machine (Linux) — on exporte donc `dataset_df` en **CSV**, à importer côté Windows. On garde le format plat (une ligne par match/joueur, comme construit) plutôt que de pré-agréger en Python : Power BI est justement fait pour construire des pivots/agrégations interactifs (taux de victoire par champion, par tranche de `killParticipation`...) sans réécrire de code à chaque nouvelle question — pré-agréger maintenant limiterait ce qu'on peut explorer ensuite.

In [8]:
import os

export_path = "../data/dataset_for_powerbi.csv"
dataset_df.to_csv(export_path, index=False)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"Exporté : {export_path} ({size_mb:.1f} Mo, {dataset_df.shape[0]} lignes, {dataset_df.shape[1]} colonnes)")

Exporté : ../data/dataset_for_powerbi.csv (58.5 Mo, 123460 lignes, 134 colonnes)


---

Export de base fait (`dataset_for_powerbi.csv`). Avant de calculer un classement de pertinence, un détour utile : vérifier qu'aucune colonne ne "triche" (fuite de donnée).

## Concept 6bis — Détecter les fuites de données (data leakage)

En creusant `maxKillDeficit` (le fait le plus corrélé après `turretTakedowns`/`turretPlatesTaken` dans un premier passage), on trouve quelque chose d'anormal : il vaut **toujours exactement 0** côté perdant, jamais côté gagnant. Ce n'est pas juste "corrélé à la victoire" — c'est **structurellement absent** d'un côté, ce qui sent la fuite de donnée : un champ qui encode presque directement le résultat plutôt qu'un vrai fait de jeu observable en cours de partie (donc inutile pour un "plan de jeu" — on ne peut pas agir dessus, il ne se révèle qu'après coup).

On vérifie si d'autres champs ont ce même défaut, systématiquement plutôt qu'au cas par cas : un champ est suspect s'il est quasiment toujours à 0 chez les perdants (< 1% de valeurs non nulles) alors qu'il est significativement présent chez les gagnants (> 10%).

In [9]:
numeric_cols_all = dataset_df.select_dtypes(include="number").columns.tolist()

loss_rows = dataset_df[~dataset_df["team_win"]]
win_rows = dataset_df[dataset_df["team_win"]]

LEAKAGE_SUSPECT_FIELDS = set()
for col in numeric_cols_all:
    loss_nonzero_pct = (loss_rows[col] != 0).mean() * 100
    win_nonzero_pct = (win_rows[col] != 0).mean() * 100
    if loss_nonzero_pct < 1 and win_nonzero_pct > 10:
        LEAKAGE_SUSPECT_FIELDS.add(col)

print(f"{len(LEAKAGE_SUSPECT_FIELDS)} champ(s) suspect(s) de fuite de donnée :", LEAKAGE_SUSPECT_FIELDS)
print("Exclu du classement de pertinence et signalé dans l'export, pour ne pas induire Power BI en erreur.")

1 champ(s) suspect(s) de fuite de donnée : {'maxKillDeficit'}
Exclu du classement de pertinence et signalé dans l'export, pour ne pas induire Power BI en erreur.


## Concept 7 — Un classement de pertinence par colonne, pour guider l'exploration Power BI

133 colonnes, c'est trop pour construire une mesure Power BI par colonne à la main. On calcule ici, en Python, une **corrélation** entre chaque fait de jeu numérique et `team_win` — une mesure simple (entre -1 et 1) de "à quel point cette colonne bouge avec la victoire". Ce n'est **pas** le même calcul que fera XGBoost/SHAP plus tard (une corrélation ne capture que des relations linéaires simples, pas les interactions entre features), mais c'est un premier tri rapide et honnête, largement suffisant pour savoir quoi explorer en priorité dans Power BI. Les champs suspects de fuite de donnée (Concept 6bis) sont exclus d'office.

On calcule aussi la moyenne de chaque colonne séparément chez les équipes qui gagnent et celles qui perdent — plus parlant qu'un simple coefficient (ex: "les équipes qui gagnent prennent en moyenne 1.8 dragon de plus" est plus concret que "corrélation = 0.34").

In [10]:
win_as_int = dataset_df["team_win"].astype(int)
numeric_cols = [c for c in numeric_cols_all if c not in LEAKAGE_SUSPECT_FIELDS]

means_when_win = dataset_df.loc[dataset_df["team_win"], numeric_cols].mean()
means_when_loss = dataset_df.loc[~dataset_df["team_win"], numeric_cols].mean()

relevance_df = pd.DataFrame({
    "feature": numeric_cols,
    "correlation_with_win": [dataset_df[col].corr(win_as_int) for col in numeric_cols],
    "mean_when_win": means_when_win.values,
    "mean_when_loss": means_when_loss.values,
})
relevance_df["abs_correlation"] = relevance_df["correlation_with_win"].abs()
relevance_df = relevance_df.sort_values("abs_correlation", ascending=False).reset_index(drop=True)

relevance_df.head(20)

/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,feature,correlation_with_win,mean_when_win,mean_when_loss,abs_correlation
0,turretTakedowns,0.600857,3.293536,0.851596,0.600857
1,turretPlatesTaken,0.500577,10.836951,4.593212,0.500577
2,kda,0.473400,5.457313,1.722855,0.473400
3,teamBaronKills,0.466307,0.841730,0.223554,0.466307
4,takedowns,0.414726,17.272185,11.002527,0.414726
5,baronTakedowns,0.393072,0.631055,0.164474,0.393072
6,deathsByEnemyChamps,-0.387469,4.683736,7.165900,0.387469
7,maxLevelLeadLaneOpponent,0.371550,1.964328,1.263241,0.371550
8,firstTurretKilled,0.370009,0.681192,0.311194,0.370009
9,goldPerMinute,0.353152,462.477575,392.113973,0.353152


In [11]:
relevance_export_path = "../data/feature_relevance.csv"
relevance_df.to_csv(relevance_export_path, index=False)
print(f"Exporté : {relevance_export_path} ({len(relevance_df)} colonnes)")

Exporté : ../data/feature_relevance.csv (128 colonnes)


## Concept 8 — État des lieux des valeurs manquantes

Certains champs `challenges` ne sont calculés que dans certaines situations (ex: `earliestDragonTakedown` n'existe que si l'équipe a pris un dragon) — d'où des `NaN` déjà aperçus dans `dataset_df.head()`. Avant d'aller plus loin, un vrai état des lieux : quelles colonnes ont le plus de valeurs manquantes, et dans quelle proportion. Utile à la fois pour Power BI (savoir qu'un champ avec 80% de `NaN` n'est pas fiable à visualiser tel quel) et pour l'entraînement à venir (XGBoost gère nativement les `NaN`, mais mieux vaut le savoir que le découvrir après coup).

In [12]:
missing_pct = (dataset_df.isna().mean() * 100).sort_values(ascending=False)
missing_df = missing_pct[missing_pct > 0].reset_index()
missing_df.columns = ["feature", "missing_pct"]

print(f"{len(missing_df)} colonnes avec au moins une valeur manquante (sur {dataset_df.shape[1]}).")
print(f"Colonnes avec plus de 50% de valeurs manquantes : {(missing_df['missing_pct'] > 50).sum()}")

missing_export_path = "../data/missing_values.csv"
missing_df.to_csv(missing_export_path, index=False)
print(f"Exporté : {missing_export_path}")

missing_df.head(15)

26 colonnes avec au moins une valeur manquante (sur 134).
Colonnes avec plus de 50% de valeurs manquantes : 17
Exporté : ../data/missing_values.csv


,feature,missing_pct
0,hadAfkTeammate,98.751822
1,earliestElderDragon,97.804957
2,fasterSupportQuestCompletion,95.230844
3,fastestLegendary,93.844160
4,teleportTakedowns,92.881095
5,highestCrowdControlScore,90.000000
6,highestChampionDamage,90.000000
7,highestWardKills,87.974243
8,soloTurretsLategame,80.226794
9,killsOnLanersEarlyJungleAsJungler,80.000000


## Concept 9 — Pertinence par rôle : le classement global cache des écarts

Le classement de pertinence du Concept 7 mélange les 5 rôles ensemble. Mais `visionScorePerMinute` compte sans doute beaucoup plus pour un support que pour un mid, alors que `damagePerMinute` c'est probablement l'inverse — la corrélation globale peut sous-estimer l'importance d'un fait qui compte énormément pour un seul rôle et peu pour les autres. On recalcule donc la même corrélation, **séparément pour chaque rôle**, et on exporte le tout dans un format long (une ligne par combinaison rôle × fait) directement exploitable dans Power BI (un visuel avec un slicer sur `role`, ou une matrice rôle × top faits).

In [13]:
role_relevance_rows = []
for role in sorted(dataset_df["role"].unique()):
    role_df = dataset_df[dataset_df["role"] == role]
    role_win = role_df["team_win"].astype(int)
    for col in numeric_cols:
        role_relevance_rows.append({
            "role": role,
            "feature": col,
            "correlation_with_win": role_df[col].corr(role_win),
        })

role_relevance_df = pd.DataFrame(role_relevance_rows)
role_relevance_df["abs_correlation"] = role_relevance_df["correlation_with_win"].abs()

role_relevance_export_path = "../data/feature_relevance_by_role.csv"
role_relevance_df.to_csv(role_relevance_export_path, index=False)
print(f"Exporté : {role_relevance_export_path} ({len(role_relevance_df)} lignes)")
print()

for role in sorted(dataset_df["role"].unique()):
    top3 = (
        role_relevance_df[role_relevance_df["role"] == role]
        .sort_values("abs_correlation", ascending=False)
        .head(3)
    )
    top3_str = ", ".join(f"{r.feature} ({r.correlation_with_win:.2f})" for r in top3.itertuples())
    print(f"{role:10s} : {top3_str}")

/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/

Exporté : ../data/feature_relevance_by_role.csv (768 lignes)

BOTTOM     : turretTakedowns (0.67), turretPlatesTaken (0.60), goldPerMinute (0.50)
JUNGLE     : turretTakedowns (0.58), kda (0.53), dragonTakedowns (0.52)
MIDDLE     : turretTakedowns (0.60), turretPlatesTaken (0.53), kda (0.47)
TOP        : turretTakedowns (0.57), turretPlatesTaken (0.48), teamBaronKills (0.47)
UNKNOWN    : enemyChampionImmobilizations (1.00), turretPlatesTaken (1.00), turretTakedowns (1.00)
UTILITY    : turretTakedowns (0.64), turretPlatesTaken (0.55), kda (0.48)


/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Concept 10 — Table de synthèse par champion

Une table prête à l'emploi pour Power BI : par champion, nombre de parties jouées, taux de victoire, et rôle principal (le plus fréquent — certains champions sont joués à plusieurs postes). Power BI peut recalculer un taux de victoire à la volée (Concept 2 du guide), mais avoir cette version pré-calculée sert de **référence de contrôle** (si les deux ne correspondent pas, on saura qu'il y a un bug quelque part) et permet de trier/filtrer immédiatement sans écrire de DAX.

In [14]:
champion_summary = dataset_df.groupby("champion").agg(
    games_played=("team_win", "count"),
    win_rate=("team_win", "mean"),
    primary_role=("role", lambda s: s.mode().iloc[0]),
).reset_index()

champion_summary = champion_summary.sort_values("games_played", ascending=False).reset_index(drop=True)

champion_summary_export_path = "../data/champion_summary.csv"
champion_summary.to_csv(champion_summary_export_path, index=False)
print(f"Exporté : {champion_summary_export_path} ({len(champion_summary)} champions)")

champion_summary.head(10)

Exporté : ../data/champion_summary.csv (173 champions)


,champion,games_played,win_rate,primary_role
0,Yunara,2355,0.515499,BOTTOM
1,Tristana,2146,0.505592,BOTTOM
2,Sylas,2101,0.500238,JUNGLE
3,Yone,2095,0.525537,MIDDLE
4,Jhin,1916,0.514614,BOTTOM
5,Viktor,1816,0.485683,MIDDLE
6,Syndra,1709,0.516091,MIDDLE
7,Lulu,1692,0.513002,UTILITY
8,Nautilus,1682,0.523781,UTILITY
9,Qiyana,1667,0.519496,JUNGLE


---

**Checkpoint final** : cinq fichiers à transférer vers la machine Windows (aucun versionné dans Git) :
- `dataset_for_powerbi.csv` — table plate (maintenant avec `role`).
- `feature_relevance.csv` — pertinence globale (fuites de données exclues).
- `feature_relevance_by_role.csv` — pertinence par rôle (format long).
- `missing_values.csv` — pourcentage de valeurs manquantes par colonne (26 colonnes concernées, 17 à plus de 50%).
- `champion_summary.csv` — parties jouées, taux de victoire, rôle principal par champion.

**Attention en explorant `feature_relevance_by_role.csv`** : le rôle `UNKNOWN` (4 lignes seulement sur 123 460, un artefact ponctuel de l'API Riot — `teamPosition` vide) affiche des corrélations de `1.00` sur plusieurs faits : c'est un pur artefact statistique d'un échantillon minuscule, pas un vrai signal. À filtrer/ignorer dans Power BI.

Le guide [`docs/powerbi_guide.md`](../docs/powerbi_guide.md) est mis à jour avec des sections dédiées à chacun de ces nouveaux fichiers.

Une fois cette exploration faite, prochaine étape technique : entraîner le modèle Gradient Boosted Trees et l'expliquer avec SHAP (en excluant `LEAKAGE_SUSPECT_FIELDS` de l'entraînement).

## Concept 11 — Séparer les causes des conséquences : se restreindre à l'early game

Une équipe qui écrase déjà l'autre accumule des tours, des kills, un meilleur KDA... **tout au long de la partie**, simplement parce qu'elle gagne déjà — c'est la contamination "conséquence" plutôt que "cause". Se restreindre aux faits **précoces** (avant ~20 min) rapproche des vraies décisions stratégiques : le snowball n'a pas encore eu le temps de tout expliquer.

La plupart des colonnes `challenges` sont des **totaux sur toute la partie**, pas découpées dans le temps (pas de Timeline collectée, voir mémoire projet). Mais certaines sont bornées dans le temps par construction, ou explicitement nommées comme précoces par Riot. On classe donc les colonnes en 3 catégories, **vérifiées** plutôt que supposées :

**A — Bornées par les mécaniques du jeu (vérifié sur le web, patch actuel)** :
- Héraut (`riftHeraldTakedowns`, `teamRiftHeraldKills`, `multiTurretRiftHeraldCount`, `dancedWithRiftHerald`) : apparaît à 15:00, disparaît à 19:45/19:55 — [confirmé, saison 2026](https://www.nerfplz.com/2026/08/every-summoners-rift-objective.html).
- Voidgrubs (`voidMonsterKill`) : apparaissent à 8:00, disparaissent à 14:45/14:55, une seule fois par partie — [même source](https://www.nerfplz.com/2026/08/every-summoners-rift-objective.html).
- `takedownsBeforeJungleMinionSpawn` : les camps de jungle apparaissent à 1:30 — mécanique de base non re-vérifiée pour 2026, mais peu susceptible d'avoir changé.

**Correction importante** : `turretPlatesTaken` avait été classé ici par erreur — hypothèse non vérifiée que les plaques disparaissent à 14 min. En vérifiant, le **patch 26.01** (sorti le 7-8 janvier 2026, donc actif pendant toute notre collecte de septembre) a **supprimé cette expiration** ([confirmé](https://www.altchar.com/game-news/league-of-legends-2026-s1-gameplay-changes-atakhan-removed-new-turret-plates-faster-minions-and-vision-upgrades-aXwcS6Y1psgZ) — vérifié aussi empiriquement : `turretPlatesTaken` atteint 41 dans nos données, impossible sous l'ancien plafond de 14 min). Ce champ est donc retiré de la catégorie A et traité comme un cumul sur toute la partie. `kTurretsDestroyedBeforePlatesFall` (quasi toujours à 0, jamais au-dessus de 5) semble être un vestige de l'ancienne mécanique — exclu aussi.

**B — Explicitement nommés comme précoces par Riot** (confiance moins totale sur la définition exacte, notamment pour les 3 marqués `*`) :
`jungleCsBefore10Minutes`, `laneMinionsFirst10Minutes`, `takedownsFirstXMinutes`, `outerTurretExecutesBefore10Minutes`, `wardTakedownsBefore20M`, `acesBefore15Minutes`, `earlyLaningPhaseGoldExpAdvantage`, `laningPhaseGoldExpAdvantage`, `maxCsAdvantageOnLaneOpponent`*, `maxLevelLeadLaneOpponent`*, `visionScoreAdvantageLaneOpponent`*, `quickFirstTurret`, `landSkillShotsEarlyGame`, `killsOnOtherLanesEarlyJungleAsLaner`, `killsOnLanersEarlyJungleAsJungler`, `junglerKillsEarlyJungle`, `getTakedownsInAllLanesEarlyJungleAsLaner`.

**C — Horodatages** (on garde la valeur brute, en secondes) : `earliestDragonTakedown`, `earliestBaron`, `earliestElderDragon`, `firstTurretKilledTime`, `fastestLegendary`, `mejaisFullStackInTime`.

**Exclus en plus des fuites déjà trouvées** : `killsOnRecentlyHealedByAramPack` (nom ARAM, aurait dû être exclu au Concept 4 — oubli corrigé ici) et `bountyGold` (l'argent de prime s'accumule surtout tard, quand quelqu'un est déjà très en avance — même logique de fuite que `maxKillDeficit`, à vérifier plus tard mais exclu par précaution).

In [15]:
MECHANICALLY_CAPPED_FIELDS = {
    "riftHeraldTakedowns", "teamRiftHeraldKills", "multiTurretRiftHeraldCount", "dancedWithRiftHerald",
    "voidMonsterKill",
    "takedownsBeforeJungleMinionSpawn",
}

EXPLICITLY_EARLY_FIELDS = {
    "jungleCsBefore10Minutes", "laneMinionsFirst10Minutes", "takedownsFirstXMinutes",
    "outerTurretExecutesBefore10Minutes", "wardTakedownsBefore20M", "acesBefore15Minutes",
    "earlyLaningPhaseGoldExpAdvantage", "laningPhaseGoldExpAdvantage",
    "maxCsAdvantageOnLaneOpponent", "maxLevelLeadLaneOpponent", "visionScoreAdvantageLaneOpponent",
    "quickFirstTurret", "landSkillShotsEarlyGame",
    "killsOnOtherLanesEarlyJungleAsLaner", "killsOnLanersEarlyJungleAsJungler",
    "junglerKillsEarlyJungle", "getTakedownsInAllLanesEarlyJungleAsLaner",
}

TIMESTAMP_FIELDS = {
    "earliestDragonTakedown", "earliestBaron", "earliestElderDragon",
    "firstTurretKilledTime", "fastestLegendary", "mejaisFullStackInTime",
}

NOT_CAUSAL_FIELDS = {
    "killsOnRecentlyHealedByAramPack",  # oubli du filtre ARAM (Concept 4)
    "bountyGold",  # s'accumule tard, quand quelqu'un est déjà très en avance
    "turretPlatesTaken",  # patch 26.01 : n'expire plus à 14 min, cumul sur toute la partie
    "kTurretsDestroyedBeforePlatesFall",  # vestige de l'ancienne mécanique, quasi toujours 0
}

EARLY_GAME_FIELDS = MECHANICALLY_CAPPED_FIELDS | EXPLICITLY_EARLY_FIELDS | TIMESTAMP_FIELDS

# sanity check : tous ces noms existent-ils vraiment dans nos colonnes ?
all_referenced = EARLY_GAME_FIELDS | NOT_CAUSAL_FIELDS
unknown_fields = all_referenced - set(numeric_cols_all)
print("Champs référencés mais absents des colonnes (faute de frappe potentielle):", unknown_fields or "aucun")

print()
print(f"Champs 'early game' retenus : {len(EARLY_GAME_FIELDS)}")
print(f"Champs exclus en plus des fuites (Concept 6bis) : {len(NOT_CAUSAL_FIELDS)}")

Champs référencés mais absents des colonnes (faute de frappe potentielle): aucun

Champs 'early game' retenus : 29
Champs exclus en plus des fuites (Concept 6bis) : 4


On recalcule le classement de pertinence (même méthode que Concept 7), mais restreint à `EARLY_GAME_FIELDS` uniquement — c'est ce sous-ensemble, pas le classement global, qui devrait guider la suite (choix des features d'entraînement, et exploration Power BI orientée "actionnable").

In [16]:
early_cols = [c for c in EARLY_GAME_FIELDS if c not in LEAKAGE_SUSPECT_FIELDS]

means_when_win_early = dataset_df.loc[dataset_df["team_win"], early_cols].mean()
means_when_loss_early = dataset_df.loc[~dataset_df["team_win"], early_cols].mean()

early_relevance_df = pd.DataFrame({
    "feature": early_cols,
    "correlation_with_win": [dataset_df[col].corr(win_as_int) for col in early_cols],
    "mean_when_win": means_when_win_early.values,
    "mean_when_loss": means_when_loss_early.values,
})
early_relevance_df["abs_correlation"] = early_relevance_df["correlation_with_win"].abs()
early_relevance_df = early_relevance_df.sort_values("abs_correlation", ascending=False).reset_index(drop=True)

early_relevance_export_path = "../data/feature_relevance_early_game.csv"
early_relevance_df.to_csv(early_relevance_export_path, index=False)
print(f"Exporté : {early_relevance_export_path} ({len(early_relevance_df)} champs)")

early_relevance_df

Exporté : ../data/feature_relevance_early_game.csv (29 champs)


/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmed/anaconda3/envs/lol-assistant/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,feature,correlation_with_win,mean_when_win,mean_when_loss,abs_correlation
0,maxLevelLeadLaneOpponent,0.371550,1.964328,1.263241,0.371550
1,teamRiftHeraldKills,0.266788,0.537826,0.275717,0.266788
2,voidMonsterKill,0.247976,1.299287,0.639252,0.247976
3,takedownsFirstXMinutes,0.247212,6.105929,4.501442,0.247212
4,maxCsAdvantageOnLaneOpponent,0.189785,29.767251,19.471451,0.189785
5,laningPhaseGoldExpAdvantage,0.183587,0.191303,0.067968,0.183587
6,riftHeraldTakedowns,0.133480,0.198526,0.102997,0.133480
7,visionScoreAdvantageLaneOpponent,0.129668,0.191123,0.026262,0.129668
8,killsOnLanersEarlyJungleAsJungler,0.108915,1.508181,1.236433,0.108915
9,earlyLaningPhaseGoldExpAdvantage,0.101976,0.124790,0.065016,0.101976


---

**Checkpoint final (mis à jour)** : sixième fichier à transférer, `feature_relevance_early_game.csv` — le classement de pertinence restreint aux faits précoces/causaux, celui qui compte vraiment pour la suite.

**Ce que ça change pour l'entraînement** (prochaine session) : la liste de features utilisée pour XGBoost ne sera pas les ~128 colonnes disponibles, mais `EARLY_GAME_FIELDS` (moins `LEAKAGE_SUSPECT_FIELDS`) — un sous-ensemble volontairement restreint, pour que le modèle apprenne des leviers activables tôt en partie plutôt que des symptômes d'une victoire déjà acquise.